# Credit derivatives (CDS)

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_across_asset_classes.ipynb`. Single-name **`credit_default_swap`** with a **`HazardCurve`**; model key **`hazard_rate`**.


## Concept

- **Premium leg** vs **protection leg**; survival built from **hazard**.
- **CS01** bumps spreads / hazard to summarize credit DV01.
- **Survival** \(S(t)\) links to cumulative hazard — ask for registry metrics that expose it when available.


In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics, usd_ois_curve
from finstack_quant.core.market_data import HazardCurve, MarketContext
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    price_instrument_with_metrics,
    validate_instrument_json,
)

print("Imports OK (CDS).")


Imports OK (CDS).


## Instrument JSON


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

cds = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-CORP-5Y",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": "2025-03-20",
                "end": "2030-03-20",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": "usny",
                "day_count": "Act360",
                "spread_bp": 100.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-HAZARD",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

validate_instrument_json(cds)
print("CDS JSON OK")


CDS JSON OK


## MarketContext


In [3]:
ois = usd_ois_curve(AS_OF)
hazard = HazardCurve(
    "CORP-HAZARD",
    AS_OF,
    [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
    recovery_rate=0.40,
)
mc = MarketContext().insert(ois).insert(hazard)
market_json = mc.to_json()
print("hazard id CORP-HAZARD inserted")


hazard id CORP-HAZARD inserted


## Pricing


In [4]:
out = price_instrument_with_metrics(cds, market_json, AS_OF_STR, model="hazard_rate")
print(ValuationResult.from_json(out))


ValuationResult(id="CDS-CORP-5Y", price=205064.1853, currency=USD, metrics=0)


## Metrics


In [5]:
metrics_req = ["cs01", "cs01_hazard", "jump_to_default"]
out = price_instrument_with_metrics(
    cds, market_json, AS_OF_STR, model="hazard_rate", metrics=metrics_req
)
vr = ValuationResult.from_json(out)
print_metrics(vr, metrics_req)
print("Survival: decreasing function of integrated hazard; CS01 links PV to credit spread bumps.")


  cs01: 2508.4786969773995
  cs01_hazard: 2508.4786969773995
  jump_to_default: 6000000.0
Survival: decreasing function of integrated hazard; CS01 links PV to credit spread bumps.


## Takeaways

- **Model key `hazard_rate`** is the CDS default in the Python bindings.
- **`CORP-HAZARD`** (hazard) and **`USD-OIS`** (discount) must both exist in the market JSON.
- Credit metrics vary by registry wiring; missing metrics are omitted, not errored.
